In [ ]:
import random
import numpy as np
from datasets import load_dataset, Dataset

# ============================================================================
# 🧠 AI 코딩 튜터의 친절한 안내서: 위키피디아 구조 파악하기!
# ============================================================================
# ✨ 데이터셋 이름: lcw99/wikipedia-korean-20240501
# ✨ 데이터셋 의미: 2024년 5월 1일자 한국어 위키피디아 기사 모음입니다.
# ✨ 데이터의 구조: 단순히 긴 텍스트(text)로만 되어 있는 것이 아니라,
#                  '제목(title)', '소제목 리스트(section_titles)', '소제목별 내용 리스트(section_texts)'
#                  같은 구조화된 메타데이터가 포함되어 있어, AI가 데이터를 이해하기 훨씬 좋습니다!
# 🚀 오늘 목표: 이 구조화된 데이터를 활용하여, 마치 LLM(대규모 언어 모델)에게
#         '이 기사의 요약과 주요 목차를 파악해 줘'라고 지시하는 프롬프트 형태를 만들어보는 거예요!
# ============================================================================

# --- 설정값 정의 ---
DATASET_NAME = "lcw99/wikipedia-korean-20240501"
SAMPLE_COUNT = 10  # 초보자 실습이므로, 전체가 아닌 상위 10개 샘플만 사용합니다. (메모리 절약!)

# ----------------------------------------------------------------------------
# 📚 1단계: 데이터셋 로드 (메모리와 효율성을 고려한 스트리밍 방식)
# ----------------------------------------------------------------------------
print("=" * 80)
print("🌟 1단계: 위키피디아 데이터셋 로드 시도 (Streaming 모드 우선)")
print(f"✨ 데이터셋 이름: {DATASET_NAME}")
print("ℹ️ Tip: 스트리밍(streaming=True)은 데이터를 메모리에 한 번에 모두 올리지 않아 매우 큰 데이터셋 처리 시 효율적이에요!")
print("=" * 80)

dataset = None
try:
    # ⭐️ 1. 스트리밍 모드를 사용하여 데이터셋을 로드합니다. (가장 효율적인 방법)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 데이터셋을 스트리밍 모드로 성공적으로 로드했습니다! 이제 메모리 부담 없이 빠르게 탐색할 수 있어요.")
except Exception as e:
    # ⭐️ 2. 스트리밍 모드 로드에 실패했을 경우 (환경 문제 등), 일반 로드를 시도합니다.
    print(f"⚠️ 스트리밍 로드 중 오류 발생: {e}. 일반 Dataset 모드로 전환하여 로드합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 일반 Dataset 모드로 성공적으로 로드했습니다.")
    except Exception as e_fallback:
        print(f"🚨 죄송합니다! 데이터셋 로드에 실패했습니다. 환경을 확인해주세요. ({e_fallback})")
        exit()


# ----------------------------------------------------------------------------
# 🛠️ 2단계: 샘플링 및 데이터 처리 준비
# ----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("🚀 2단계: 상위 샘플을 추출하고 데이터 구조를 파악해 봅시다.")
print("=" * 80)

# 🚨 주의! streaming 모드에서는 len()을 쓸 수 없어요!
# 따라서 .take()를 사용하여 원하는 개수만큼의 반복자(iterator)를 만들고,
# 이를 리스트로 변환하여 사용하겠습니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print(f"✅ 스트리밍 모드 감지: 상위 {SAMPLE_COUNT}개의 샘플만 가져옵니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    # 최종적으로 처리하기 쉬운 리스트 형태로 변환합니다.
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 Dataset 모드입니다.
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))

if not sample_data_list:
    print("🚨 샘플 데이터를 가져올 수 없습니다. 프로그램을 종료합니다.")
    exit()

print(f"✅ 상위 {len(sample_data_list)}개의 샘플 데이터를 준비했습니다.")


# ----------------------------------------------------------------------------
# 🧠 3단계: 구조화된 데이터를 활용한 'AI 프롬프트 생성' 실습
# ----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("🧠 3단계: AI가 이해하기 쉬운 구조화된 프롬프트 생성하기 (실습 시간!)")
print("=" * 80)

print("👇 목표: 이 기사를 요약하고 핵심 목차를 추출하는 Prompt를 만들어 봅시다.")
print("-" * 80)

for i, sample in enumerate(sample_data_list):
    if i >= 5: # 너무 길어지지 않게 5개만 출력합니다.
        break
        
    title = sample.get('title', '제목 없음')
    section_titles = sample.get('section_titles', [])
    section_texts = sample.get('section_texts', [])

    # 💡 구조화된 데이터는 리스트 형태로 와요. titles와 texts는 같은 순서로 매칭됩니다.
    # 따라서 동시에 순회(zip)하는 것이 중요합니다!
    
    print(f"\n\n=== [샘플 {i+1} / {len(sample_data_list)}] ✨ 제목: {title[:40]}...")
    
    # 1. 핵심 정보 추출 (제목과 텍스트의 매칭)
    print("🔑 1. 구조화된 정보 파악 (Section Analysis)")
    
    # section_titles와 section_texts가 길이가 다를 수 있으므로, 최소 길이를 기준으로 합니다.
    min_len = min(len(section_titles), len(section_texts))
    
    if min_len > 0:
        for idx in range(min_len):
            # 소제목과 그 소제목의 내용을 함께 출력하여 구조를 파악합니다.
            section_title = section_titles[idx]
            section_text = section_texts[idx]
            
            # 텍스트가 너무 길면 앞부분만 잘라서 보여주면 깔끔해요!
            snippet = section_text[:80].replace('\n', ' ') + '...'
            print(f"  -> 📚 섹션 '{section_title}' | 내용 미리보기: {snippet}")
        
        print("   (👉 분석: 구조화된 데이터가 각 목차별로 어떤 내용을 담고 있는지 알려주어, AI가 자료를 분할 처리하기 매우 좋습니다.)")
    else:
        print("  -> ⚠️ 구조화된 목차(section) 정보를 찾을 수 없습니다. 전체 텍스트로만 분석해야 할 수도 있어요.")


    # 2. LLM 프롬프트 작성 (AI 활용 시나리오)
    print("\n✨ 2. LLM 프롬프트 생성 시나리오 (AI Prompt Generation)")
    
    # 구조화된 정보를 사용하여 LLM에게 질문할 '명령어'를 만듭니다.
    prompt = f"""
    [명령] 당신은 전문 지식 요약가입니다. 다음 위키피디아 기사를 분석하여, 
    1. 기사의 핵심 주제를 3가지 키워드로 뽑아주세요.
    2. '{title}'의 모든 소제목(section_titles)을 목차 순서대로 나열해 주세요.
    3. 각 목차별로 내용의 핵심을 한 문장으로 요약해주세요.
    
    --- 기사 전문 ---
    [FULL_TEXT]
    {sample.get('text', '내용 없음')}
    """
    
    print("----------------------------------------------------------------------------------------------------")
    print("🧠 생성된 프롬프트 (LLM에게 전달할 최종 명령문):")
    print(prompt.strip())
    print("----------------------------------------------------------------------------------------------------")
    print("✨ Tip: 이렇게 구조화된 프롬프트를 사용하면, AI는 '무엇을 해야 하는지'와 '어디서 정보를 찾아야 하는지'를 정확히 알 수 있어요!")

print("\n\n🎉👏 실습 완료! 👏🎉")
print("축하합니다! 여러분은 단순히 데이터를 읽는 것을 넘어, 이 데이터를 '어떻게 활용할지'까지 생각하는 방법을 배웠어요.")
print("다음 단계에서는 이 프롬프트를 이용하여 실제로 질문-답변(Q&A) 모델을 만들어 볼 수 있을 거예요. 정말 대단해요!")